# 損失関数と勾配降下法

損失関数と勾配降下法は、損失が下がる理由を『式』と『数値変化』の両方で確認する構成です。


## 背景と目的

学習は「何を最小化するか」で挙動が決まり、損失設計の誤りは性能低下を直接招きます。

損失関数と勾配の関係を理解すると、発散や停滞の原因を式レベルで診断できます。

更新式と学習曲線を対応付けて確認します。


## 最初に解きたい疑問

1. 損失は具体的に何を測っていて、数値が小さいと何がうれしいのか。
2. 1回の更新で loss が下がったら、何が正しくて何がまだ未確認なのか。
3. 学習率が大きすぎると、なぜ発散したり行ったり来たりするのか。
4. L2 正則化を足すと、重みや予測にどんな変化が起きるのか。
5. ミニバッチにすると、なぜ計算しやすくなり、逆に何がぶれやすくなるのか。


## 先に押さえる言葉

- `損失関数`: 予測の外れ具合を数値にしたものです。学習はこの値を小さくする方向に進みます。
- `勾配降下法`: 損失が下がる方向へ少しずつパラメータを動かす方法です。更新量は勾配と学習率で決まります。
- `学習率`: 1回の更新でどれだけ動くかを決める係数です。大きすぎると不安定になり、小さすぎると進みが遅くなります。
- `正則化`: 重みが大きくなりすぎるのを抑えるための追加項です。訓練データへの過適合を抑える目的で使います。


## 実行前の見取り図

1. 初期損失と 1 ステップ後の損失が、実際に下がっているかを見る。
2. 更新前後の重み `w_new` と勾配 `grad` の符号が、意図した方向になっているかを見る。
3. バッチごとの損失計算で、各サンプルの値が平均にどう効いているか確認する。


## つまずきやすい点

- `-(y log p + (1-y) log(1-p))` の各項が、正例と負例でどう意味を分けているかの説明が足りない。
- ミニバッチ平均がなぜ SGD のばらつきと計算効率の両方に関わるのか、直感の説明が足りない。


In [ ]:
import math


## 観察 1: 損失計算の初期値を作る

勾配降下法を理解しやすくするため、損失を計算できる初期状態を固定します。


In [ ]:
x = [1.0, -0.5, 0.3]
w = [0.2, -0.4, 0.6]
b = 0.12
z = sum(xi * wi for xi, wi in zip(x, w)) + b
y = 1 / (1 + math.exp(-z))
print('task = loss-and-gradient-descent', 'pred=', round(y, 6))


この初期予測を使って、損失の勾配と更新方向を確認します。



## 観察 2: 損失を計算する

次に、予測値に対する損失を計算します。学習は損失を下げる方向に進むので、損失の意味を言葉で説明できることが重要です。


In [ ]:
target = 1.0
eps = 1e-9
loss = -(target * math.log(y + eps) + (1 - target) * math.log(1 - y + eps))
print('loss =', round(loss, 6))
print('error =', round(target - y, 6))

損失は『どれだけ外したか』を測る物差しです。物差しがない状態では、改善の方向を決められません。



## 計算の対応表

1. $z = W x + b$
2. $\theta \leftarrow \theta - \eta \, \nabla_{\theta} L$

3. BCE + sigmoid のとき、出力前活性 `z` に対する勾配は $y - target$


## 観察 3: 勾配降下法の一歩

ここで、重みを一回更新する最小実験を行います。更新前後の損失を比較して、学習の方向が正しいかを確認します。


In [ ]:
lr = 0.1
grad_z = y - target  # BCE + sigmoid のとき dL/dz
w_new = [wi - lr * grad_z * xi for wi, xi in zip(w, x)]
b_new = b - lr * grad_z
z_new = sum(xi * wi for xi, wi in zip(x, w_new)) + b_new
y_new = 1 / (1 + math.exp(-z_new))
print('grad_z =', round(grad_z, 6))
print('b before/after =', round(b, 6), round(b_new, 6))
print('y before/after =', round(y, 6), round(y_new, 6))


この更新で損失が下がれば、勾配方向が合理的だったと言えます。下がらないなら学習率や符号を疑います。



ここでは重みだけでなく bias も同時に更新し、BCE と勾配の対応が崩れないようにしています。


## 観察 4: 正則化の感覚を作る

次に、重みが大きくなりすぎることを抑える正則化項を追加します。過学習対策をコードで体験するのが狙いです。


In [ ]:
l2 = 0.01
weight_sq = sum(wi * wi for wi in w_new)
y_reg = 1 / (1 + math.exp(-(sum(xi * wi for xi, wi in zip(x, w_new)) + b_new)))
data_loss_reg = -(target * math.log(y_reg + eps) + (1 - target) * math.log(1 - y_reg + eps))
loss_reg = data_loss_reg + l2 * weight_sq
print('data_loss@updated =', round(data_loss_reg, 6))
print('weight_sq =', round(weight_sq, 6))
print('regularized loss =', round(loss_reg, 6))

ここでは損失に penalty が足されることだけ確認しています。正則化が更新式そのものをどう変えるかは、後続の `optimization-regularization` ノートで詳しく扱います。



## 観察 5: ミニバッチで平均損失と平均勾配を見る

最後に、複数サンプルをまとめて扱う発想を確認します。ここでは各サンプルの損失と勾配を平均してから 1 回だけ更新します。


In [ ]:
batch = [[0.8, -0.4, 0.2], [0.2, 0.1, -0.3], [0.5, -0.2, 0.7]]
targets = [1.0, 0.0, 1.0]
eps = 1e-7
sample_preds = []
sample_losses = []
sample_grad_z = []

for bx, bt in zip(batch, targets):
    z_b = sum(xi * wi for xi, wi in zip(bx, w_new)) + b_new
    y_b = 1 / (1 + math.exp(-z_b))
    loss_b = -(bt * math.log(y_b + eps) + (1 - bt) * math.log(1 - y_b + eps))
    sample_preds.append(y_b)
    sample_losses.append(loss_b)
    sample_grad_z.append(y_b - bt)

grad_w_batch = [
    sum(g * bx[j] for g, bx in zip(sample_grad_z, batch)) / len(batch)
    for j in range(len(w_new))
]
grad_b_batch = sum(sample_grad_z) / len(batch)

w_batch = [wi - lr * gw for wi, gw in zip(w_new, grad_w_batch)]
b_batch = b_new - lr * grad_b_batch

updated_losses = []
for bx, bt in zip(batch, targets):
    z_b = sum(xi * wi for xi, wi in zip(bx, w_batch)) + b_batch
    y_b = 1 / (1 + math.exp(-z_b))
    updated_losses.append(-(bt * math.log(y_b + eps) + (1 - bt) * math.log(1 - y_b + eps)))

print('sample_preds =', [round(p, 4) for p in sample_preds])
print('sample_losses =', [round(v, 4) for v in sample_losses])
print('batch_loss_mean =', round(sum(sample_losses) / len(sample_losses), 6))
print('grad_b_batch =', round(grad_b_batch, 6))
print('grad_w_batch =', [round(v, 6) for v in grad_w_batch])
print('batch_loss_mean_after =', round(sum(updated_losses) / len(updated_losses), 6))


ミニバッチでは各サンプルの損失と勾配を平均してから 1 回更新します。1 件ずつの SGD よりぶれが減り、完全バッチより計算を小分けにできるので、その中間としてよく使われます。



## 要点整理

今回のノートで押さえておくべき誤解しやすい点を整理します。

1. 学習率が大きすぎて発散する
2. 検証損失の監視をせず過学習を見逃す
3. 前処理と活性化の相性を無視する
